In [ ]:
# Click & Collect: 15-second vertical product showcase (no SVD-XT landscape clip)
# Paste a clean, public product-photo URL below, then Run All once.
IMAGE_URL = "PASTE_CLEAN_PUBLIC_PRODUCT_IMAGE_URL_HERE"
DURATION_SECONDS = 15
OUTPUT = "/kaggle/working/click_collect_vertical_15s.mp4"

import os, subprocess, urllib.request
from PIL import Image, ImageFilter, ImageEnhance
from io import BytesIO

if not IMAGE_URL.startswith('https://'):
    raise ValueError('IMAGE_URL must be a public HTTPS image URL.')
os.makedirs('/kaggle/working', exist_ok=True)
req = urllib.request.Request(IMAGE_URL, headers={'User-Agent':'Mozilla/5.0'})
with urllib.request.urlopen(req, timeout=60) as r:
    source = Image.open(BytesIO(r.read())).convert('RGB')
print('Source image:', source.size)

# Center-crop to portrait 9:16. For the current blender source this keeps
# the centered product and trims the unrelated top-right supplier text.
target_ratio = 9 / 16
w, h = source.size
if w / h > target_ratio:
    crop_w = int(h * target_ratio)
    left = max(0, (w - crop_w) // 2)
    source = source.crop((left, 0, left + crop_w, h))
else:
    crop_h = int(w / target_ratio)
    top = max(0, (h - crop_h) // 2)
    source = source.crop((0, top, w, top + crop_h))
source = ImageEnhance.Sharpness(source).enhance(1.12)
source = source.resize((1080, 1920), Image.Resampling.LANCZOS)
still = '/kaggle/working/product_portrait_1080x1920.jpg'
source.save(still, quality=95, optimize=True)

# A restrained 15-second camera push keeps the real product pixels stable.
# This is a polished motion-graphics showcase, NOT generative AI footage.
fps = 30
frames = DURATION_SECONDS * fps
vf = (f"zoompan=z='min(zoom+0.00012,1.055)':d={frames}:s=1080x1920:fps={fps},"
      "format=yuv420p")
cmd = ['ffmpeg','-y','-loop','1','-i',still,'-vf',vf,'-t',str(DURATION_SECONDS),
       '-r',str(fps),'-an','-c:v','libx264','-preset','slow','-crf','18',
       '-pix_fmt','yuv420p','-movflags','+faststart',OUTPUT]
run = subprocess.run(cmd, capture_output=True, text=True)
if run.returncode:
    raise RuntimeError(run.stderr[-3000:])
probe = subprocess.run(['ffprobe','-v','error','-show_entries','stream=width,height,r_frame_rate','-show_entries','format=duration','-of','default=noprint_wrappers=1',OUTPUT],capture_output=True,text=True)
print('VIDEO READY:', OUTPUT)
print(probe.stdout)
print('Download click_collect_vertical_15s.mp4 from the Kaggle Output panel.')
